In [1]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [2]:
bureau = pl.read_csv("../data/raw/bureau.csv")

bureau.head()

SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
i64,i64,str,str,i64,i64,f64,f64,f64,i64,f64,f64,f64,f64,str,i64,str
215354,5714462,"""Closed""","""currency 1""",-497,0,-153.0,-153.0,null,0,91323.0,0.0,null,0.0,"""Consumer credit""",-131,null
215354,5714463,"""Active""","""currency 1""",-208,0,1075.0,null,null,0,225000.0,171342.0,null,0.0,"""Credit card""",-20,null
215354,5714464,"""Active""","""currency 1""",-203,0,528.0,null,null,0,464323.5,null,null,0.0,"""Consumer credit""",-16,null
215354,5714465,"""Active""","""currency 1""",-203,0,null,null,null,0,90000.0,null,null,0.0,"""Credit card""",-16,null
215354,5714466,"""Active""","""currency 1""",-629,0,1197.0,null,77674.5,0,2.7e6,null,null,0.0,"""Consumer credit""",-21,null


In [3]:
bureau.schema

Schema([('SK_ID_CURR', Int64),
        ('SK_ID_BUREAU', Int64),
        ('CREDIT_ACTIVE', String),
        ('CREDIT_CURRENCY', String),
        ('DAYS_CREDIT', Int64),
        ('CREDIT_DAY_OVERDUE', Int64),
        ('DAYS_CREDIT_ENDDATE', Float64),
        ('DAYS_ENDDATE_FACT', Float64),
        ('AMT_CREDIT_MAX_OVERDUE', Float64),
        ('CNT_CREDIT_PROLONG', Int64),
        ('AMT_CREDIT_SUM', Float64),
        ('AMT_CREDIT_SUM_DEBT', Float64),
        ('AMT_CREDIT_SUM_LIMIT', Float64),
        ('AMT_CREDIT_SUM_OVERDUE', Float64),
        ('CREDIT_TYPE', String),
        ('DAYS_CREDIT_UPDATE', Int64),
        ('AMT_ANNUITY', String)])

In [4]:
bureau_balance = pl.read_csv("../data/raw/bureau_balance.csv")

bureau_balance.head()

SK_ID_BUREAU,MONTHS_BALANCE,STATUS
i64,i64,str
5715448,0,"""C"""
5715448,-1,"""C"""
5715448,-2,"""C"""
5715448,-3,"""C"""
5715448,-4,"""C"""


In [5]:
bureau_balance.schema


Schema([('SK_ID_BUREAU', Int64),
        ('MONTHS_BALANCE', Int64),
        ('STATUS', String)])

In [6]:
bb_numeric = bureau_balance.with_columns([
    pl.col("STATUS")
    .replace({"C":0, "X":0})
    .cast(pl.Int64,strict=False)
    .fill_null(0)
    .alias("STATUS_NUMERICAL")
])

bb_features = (
    bb_numeric
    .group_by("SK_ID_BUREAU")
    .agg([
        pl.len().alias("bb_total_months_tracked"),
        pl.col("STATUS_NUMERICAL").max().alias("bb_worst_status"),
        pl.col("STATUS_NUMERICAL").filter(pl.col("STATUS_NUMERICAL") > 0).len().alias("bb_months_overdue_count"),
        pl.col("STATUS_NUMERICAL").filter(pl.col("MONTHS_BALANCE") >= -6).max().alias("bb_worst_status_last_6m")
    ])
)

bb_features.head()  


SK_ID_BUREAU,bb_total_months_tracked,bb_worst_status,bb_months_overdue_count,bb_worst_status_last_6m
i64,u32,i64,u32,i64
6707231,18,0,0,0
6195629,12,0,0,0
5046462,83,0,0,0
6013833,51,1,8,0
5242293,22,0,0,0


In [7]:
bureau_combined = bureau.join(bb_features, on="SK_ID_BUREAU", how="left")

bureau_final_features = (
    bureau_combined
    .group_by("SK_ID_CURR")
    .agg([pl.len().alias("bureau_loan_count"),
          pl.col("CREDIT_TYPE").n_unique().alias("bureau_loan_types"),
          
          pl.col("DAYS_CREDIT").max().alias("bureau_days_since_last_loan"),
          pl.col("DAYS_CREDIT").min().alias("bureau_days_since_first_loan"),
          
          pl.col("AMT_CREDIT_SUM").mean().alias("bureau_avg_credit"),
          pl.col("AMT_CREDIT_SUM_DEBT").sum().alias("total_debt"),
          pl.col("AMT_CREDIT_SUM").sum().alias("total_credit"),
          
          pl.col("AMT_CREDIT_SUM_DEBT").filter(pl.col("CREDIT_ACTIVE") == "Active").sum().alias("bureau_active_debt"),
          (pl.col("AMT_CREDIT_SUM_DEBT").sum() / (pl.col("AMT_CREDIT_SUM").sum() + 1e-5)).alias("debt_to_credit_ratio"),
          
          pl.col("CREDIT_DAY_OVERDUE").max().alias("bureau_max_overdue"),
          pl.col("AMT_CREDIT_SUM_OVERDUE").sum().alias("bureau_total_amt_overdue"),
          pl.col("CNT_CREDIT_PROLONG").sum().alias("bureau_total_prolong_count"),
          
          pl.col("bb_worst_status").max().fill_null(0).alias("bureau_global_worst_status"),
          pl.col("bb_months_overdue_count").sum().fill_null(0).alias("bureau_total_months_overdue_across_loans"),
          pl.col("bb_worst_status_last_6m").max().fill_null(0).alias("bureau_global_worst_status_last_6m")])
)

bureau_final_features.head()

SK_ID_CURR,bureau_loan_count,bureau_loan_types,bureau_days_since_last_loan,bureau_days_since_first_loan,bureau_avg_credit,total_debt,total_credit,bureau_active_debt,debt_to_credit_ratio,bureau_max_overdue,bureau_total_amt_overdue,bureau_total_prolong_count,bureau_global_worst_status,bureau_total_months_overdue_across_loans,bureau_global_worst_status_last_6m
i64,u32,u32,i64,i64,f64,f64,f64,f64,f64,i64,f64,i64,i64,u32,i64
316435,10,2,-224,-2786,595634.4,2.328183e6,5.956344e6,2.328183e6,0.390875,0,0.0,0,0,0,0
180906,10,4,-378,-1974,1.5219e6,7.792749e6,1.5219e7,7.792749e6,0.51205,0,0.0,0,0,0,0
182990,4,2,-120,-1644,198107.325,486219.015,792429.3,486219.015,0.61358,0,0.0,0,0,0,0
179072,2,1,-908,-2777,128074.5,0.0,256149.0,0.0,0.0,0,0.0,0,0,0,0
267286,3,2,-421,-2732,249000.0,49324.5,747000.0,49324.5,0.06603,0,0.0,0,0,0,0


In [8]:
installment_payment = pl.read_csv("../data/raw/installments_payments.csv")
installment_payment.head()

SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
i64,i64,f64,i64,f64,f64,f64,f64
1054186,161674,1.0,6,-1180.0,-1187.0,6948.36,6948.36
1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2085231,193053,2.0,1,-63.0,-63.0,25425.0,25425.0
2452527,199697,1.0,3,-2418.0,-2426.0,24350.13,24350.13
2714724,167756,1.0,2,-1383.0,-1366.0,2165.04,2160.585


In [9]:
installment_payment.schema

Schema([('SK_ID_PREV', Int64),
        ('SK_ID_CURR', Int64),
        ('NUM_INSTALMENT_VERSION', Float64),
        ('NUM_INSTALMENT_NUMBER', Int64),
        ('DAYS_INSTALMENT', Float64),
        ('DAYS_ENTRY_PAYMENT', Float64),
        ('AMT_INSTALMENT', Float64),
        ('AMT_PAYMENT', Float64)])

In [10]:
ins_processed = installment_payment.with_columns(
    (pl.col("AMT_PAYMENT") - pl.col("AMT_INSTALMENT")).alias("ins_pay_diff"),
    (pl.col("AMT_PAYMENT") / pl.col("AMT_INSTALMENT") + 1e-5).alias("ins_pay_ratio"),
    (pl.col("DAYS_ENTRY_PAYMENT") - pl.col("DAYS_INSTALMENT")).alias("ins_day_late")
)

ins_processed.head()

SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,ins_pay_diff,ins_pay_ratio,ins_day_late
i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64
1054186,161674,1.0,6,-1180.0,-1187.0,6948.36,6948.36,0.0,1.00001,-7.0
1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525,0.0,1.00001,0.0
2085231,193053,2.0,1,-63.0,-63.0,25425.0,25425.0,0.0,1.00001,0.0
2452527,199697,1.0,3,-2418.0,-2426.0,24350.13,24350.13,0.0,1.00001,-8.0
2714724,167756,1.0,2,-1383.0,-1366.0,2165.04,2160.585,-4.455,0.997952,17.0


In [11]:
ins_features = (
    ins_processed
    .group_by("SK_ID_CURR")
    .agg([
        pl.len().alias("ins_total_installment_tracked"),
        
        pl.col("ins_pay_diff").min().alias("ins_worst_underpay"),
        pl.col("ins_pay_diff").mean().alias("ins_mean_payment_diff"),
        pl.col("ins_pay_ratio").mean().alias("ins_mean_payratio"),
        
        pl.col("ins_pay_diff").filter(pl.col("ins_pay_diff") < 0).len().alias("ins_undepayment_count"),
        
        pl.col("ins_day_late").max().alias("ins_worst_day_late"),
        pl.col("ins_day_late").mean().alias("ins_avg_day_late"),
        
        pl.col("ins_day_late").filter(pl.col("ins_day_late") > 0).len().alias("ins_late_count")
    ])
)

ins_features.head()

SK_ID_CURR,ins_total_installment_tracked,ins_worst_underpay,ins_mean_payment_diff,ins_mean_payratio,ins_undepayment_count,ins_worst_day_late,ins_avg_day_late,ins_late_count
i64,u32,f64,f64,f64,u32,f64,f64,u32
151139,6,0.0,0.0,1.00001,0,-9.0,-10.333333,0
406486,120,-8439.795,-814.65,0.90001,23,27.0,-4.625,30
420256,69,-12177.675,-556.323261,0.927546,10,11.0,-4.391304,5
270829,98,-8325.0,350.890255,1.026722,17,23.0,-6.408163,6
117329,8,0.0,0.0,1.00001,0,0.0,-13.375,0


In [12]:
credit_card_balance = pl.read_csv("../data/raw/credit_card_balance.csv")

credit_card_balance.head()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,str,i64,i64
2562384,378907,-6,56.97,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.0,0.0,0.0,0.0,1,0.0,1.0,35.0,"""Active""",0,0
2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.0,2250.0,2250.0,60175.08,64875.555,64875.555,1.0,1,0.0,0.0,69.0,"""Active""",0,0
1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.0,2250.0,2250.0,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,"""Active""",0,0
1389973,337855,-4,236572.11,225000,2250.0,2250.0,0.0,0.0,11795.76,11925.0,11925.0,224949.285,233048.97,233048.97,1.0,1,0.0,0.0,10.0,"""Active""",0,0
1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.89,27000.0,27000.0,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,"""Active""",0,0


In [13]:
cc_sorted = credit_card_balance.sort(["SK_ID_CURR","MONTHS_BALANCE"])

cc_features = (
    cc_sorted
    .group_by("SK_ID_CURR")
    .agg([
        pl.len().alias("cc_months_tracked"),
        
        pl.col("AMT_DRAWINGS_CURRENT").sum().alias("cc_total_drawings_amt"),
        pl.col("AMT_DRAWINGS_CURRENT").max().alias("cc_max_single_month_drawing"),
        pl.col("CNT_DRAWINGS_CURRENT").sum().alias("cc_total_drawings_count"),
        
        pl.col("AMT_DRAWINGS_ATM_CURRENT").sum().alias("cc_total_atm_drawings_amt"),
        (pl.col("AMT_DRAWINGS_ATM_CURRENT").sum() / (pl.col("AMT_DRAWINGS_CURRENT").sum() + 1e-5)).alias("cc_atm_to_total_drawing_ratio"),
        
        pl.col("AMT_PAYMENT_CURRENT").sum().alias("cc_total_paymemt"),
        (pl.col("AMT_PAYMENT_CURRENT").sum() / (pl.col("AMT_INST_MIN_REGULARITY").sum() + 1e-5)).alias("cc_pay_to_min_inst_ratio"),
        
        pl.col("AMT_BALANCE").max().alias("cc_max_balance_ever"),
        pl.col("AMT_BALANCE").last().alias("cc_current_balance"),
        pl.col("AMT_TOTAL_RECEIVABLE").last().alias("cc_current_total_receivable"),
        
        (pl.col("AMT_BALANCE").last() / (pl.col("AMT_CREDIT_LIMIT_ACTUAL").last() + 1e-5)).alias("cc_current_utilization_ratio"),
        
        pl.col("SK_DPD").max().alias("cc_max_days_past_due"),
        pl.col("SK_DPD_DEF").max().alias("cc_max_days_past_due_with_tolerance"),
        
        pl.col("SK_DPD").filter(pl.col("SK_DPD") > 0).len().alias("cc_month_overdue_count")
    ])
    
)

cc_features.head()

SK_ID_CURR,cc_months_tracked,cc_total_drawings_amt,cc_max_single_month_drawing,cc_total_drawings_count,cc_total_atm_drawings_amt,cc_atm_to_total_drawing_ratio,cc_total_paymemt,cc_pay_to_min_inst_ratio,cc_max_balance_ever,cc_current_balance,cc_current_total_receivable,cc_current_utilization_ratio,cc_max_days_past_due,cc_max_days_past_due_with_tolerance,cc_month_overdue_count
i64,u32,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,u32
100006,6,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0
100011,74,180000.0,180000.0,4,180000.0,1.0,358386.75,1.240933,189000.0,0.0,0.0,0.0,0,0,0
100013,96,571500.0,157500.0,23,571500.0,1.0,688161.24,5.315874,161420.22,0.0,0.0,0.0,1,1,1
100021,17,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0
100023,8,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0


In [14]:
pos_cash_balance = pl.read_csv("../data/raw/POS_CASH_balance.csv")

pos_cash_balance.head()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i64,i64,i64,f64,f64,str,i64,i64
1803195,182943,-31,48.0,45.0,"""Active""",0,0
1715348,367990,-33,36.0,35.0,"""Active""",0,0
1784872,397406,-32,12.0,9.0,"""Active""",0,0
1903291,269225,-35,48.0,42.0,"""Active""",0,0
2341044,334279,-35,36.0,35.0,"""Active""",0,0


In [15]:
pos_sorted = pos_cash_balance.sort(["SK_ID_CURR","MONTHS_BALANCE"])

pos_features = (
    pos_sorted
    .group_by("SK_ID_CURR")
    .agg([
        pl.len().alias("pos_month_tracked"),
        
        pl.col("CNT_INSTALMENT").max().alias("pos_max_loan_term"),
        pl.col("CNT_INSTALMENT_FUTURE").last().alias("pos_current_remaining_installments"),
        
        (pl.col("CNT_INSTALMENT_FUTURE").last() / (pl.col("CNT_INSTALMENT").last() + 1e-5)).alias("pos_remaining_installment_ratio"),
        
        pl.col("SK_DPD").max().alias("pos_max_days_past_due"),
        pl.col("SK_DPD_DEF").max().alias("pos_max_days_past_due_with_tolerance"),
        
        pl.col("SK_DPD").filter(pl.col("SK_DPD") > 0).len().alias("pos_month_overdue_count")
    ])
)

pos_features.head()

SK_ID_CURR,pos_month_tracked,pos_max_loan_term,pos_current_remaining_installments,pos_remaining_installment_ratio,pos_max_days_past_due,pos_max_days_past_due_with_tolerance,pos_month_overdue_count
i64,u32,f64,f64,f64,i64,i64,u32
100001,9,4.0,0.0,0.0,7,7,1
100002,19,24.0,6.0,0.25,0,0,0
100003,28,12.0,0.0,0.0,0,0,0
100004,4,4.0,0.0,0.0,0,0,0
100005,11,12.0,0.0,0.0,0,0,0


In [16]:
previous_application = pl.read_csv("../data/raw/previous_application.csv")

previous_application.head()

SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
i64,i64,str,f64,f64,f64,f64,f64,str,i64,str,i64,f64,f64,f64,str,str,i64,str,str,str,str,str,str,str,str,i64,str,f64,str,str,f64,f64,f64,f64,f64,f64
2030495,271877,"""Consumer loans""",1730.43,17145.0,17145.0,0.0,17145.0,"""SATURDAY""",15,"""Y""",1,0.0,0.182832,0.867336,"""XAP""","""Approved""",-73,"""Cash through the bank""","""XAP""",null,"""Repeater""","""Mobile""","""POS""","""XNA""","""Country-wide""",35,"""Connectivity""",12.0,"""middle""","""POS mobile with interest""",365243.0,-42.0,300.0,-42.0,-37.0,0.0
2802425,108129,"""Cash loans""",25188.615,607500.0,679671.0,null,607500.0,"""THURSDAY""",11,"""Y""",1,null,null,null,"""XNA""","""Approved""",-164,"""XNA""","""XAP""","""Unaccompanied""","""Repeater""","""XNA""","""Cash""","""x-sell""","""Contact center""",-1,"""XNA""",36.0,"""low_action""","""Cash X-Sell: low""",365243.0,-134.0,916.0,365243.0,365243.0,1.0
2523466,122040,"""Cash loans""",15060.735,112500.0,136444.5,null,112500.0,"""TUESDAY""",11,"""Y""",1,null,null,null,"""XNA""","""Approved""",-301,"""Cash through the bank""","""XAP""","""Spouse, partner""","""Repeater""","""XNA""","""Cash""","""x-sell""","""Credit and cash offices""",-1,"""XNA""",12.0,"""high""","""Cash X-Sell: high""",365243.0,-271.0,59.0,365243.0,365243.0,1.0
2819243,176158,"""Cash loans""",47041.335,450000.0,470790.0,null,450000.0,"""MONDAY""",7,"""Y""",1,null,null,null,"""XNA""","""Approved""",-512,"""Cash through the bank""","""XAP""",null,"""Repeater""","""XNA""","""Cash""","""x-sell""","""Credit and cash offices""",-1,"""XNA""",12.0,"""middle""","""Cash X-Sell: middle""",365243.0,-482.0,-152.0,-182.0,-177.0,1.0
1784265,202054,"""Cash loans""",31924.395,337500.0,404055.0,null,337500.0,"""THURSDAY""",9,"""Y""",1,null,null,null,"""Repairs""","""Refused""",-781,"""Cash through the bank""","""HC""",null,"""Repeater""","""XNA""","""Cash""","""walk-in""","""Credit and cash offices""",-1,"""XNA""",24.0,"""high""","""Cash Street: high""",null,null,null,null,null,null


In [17]:
prev_sorted = previous_application.sort(["SK_ID_CURR", "DAYS_DECISION"])

prev_features = (
    prev_sorted
    .group_by("SK_ID_CURR")
    .agg([
        pl.len().alias("prev_total_applications"),
        
        pl.col("DAYS_DECISION").max().alias("prev_days_since_last_decision"),
        pl.col("DAYS_DECISION").min().alias("prev_days_since_first_decision"),
        
        pl.col("AMT_APPLICATION").sum().alias("prev_total_amt_asked"),
        pl.col("AMT_CREDIT").sum().alias("prev_total_amt_approved"),
        (pl.col("AMT_CREDIT").sum() / (pl.col("AMT_APPLICATION").sum() + 1e-5)).alias("prev_credit_to_application_ratio"),
        pl.col("AMT_ANNUITY").mean().alias("prev_avg_annuity_amt"),
        pl.col("RATE_DOWN_PAYMENT").mean().alias("prev_avg_down_payment_rate"),
        
        pl.col("CNT_PAYMENT").mean().alias("prev_avg_loan_term_months"),
        
        pl.col("NAME_CONTRACT_STATUS").filter(pl.col("NAME_CONTRACT_STATUS") == "Refused").len().alias("prev_count_refused"),
        pl.col("NAME_CONTRACT_STATUS").filter(pl.col("NAME_CONTRACT_STATUS") == "Approved").len().alias("prev_count_approved"),
        pl.col("NAME_CONTRACT_STATUS").filter(pl.col("NAME_CONTRACT_STATUS") == "Canceled").len().alias("prev_count_canceled"),
        
        (pl.col("NAME_CONTRACT_STATUS").filter(pl.col("NAME_CONTRACT_STATUS") == "Refused").len() / pl.len()).alias("prev_rejection_rate"),
        pl.col("NAME_CONTRACT_STATUS").last().alias("prev_most_recent_status")
    ])
)

prev_features = prev_features.to_dummies(columns=["prev_most_recent_status"])

prev_features.head()

SK_ID_CURR,prev_total_applications,prev_days_since_last_decision,prev_days_since_first_decision,prev_total_amt_asked,prev_total_amt_approved,prev_credit_to_application_ratio,prev_avg_annuity_amt,prev_avg_down_payment_rate,prev_avg_loan_term_months,prev_count_refused,prev_count_approved,prev_count_canceled,prev_rejection_rate,prev_most_recent_status_Approved,prev_most_recent_status_Canceled,prev_most_recent_status_Refused,prev_most_recent_status_Unused offer
i64,u32,i64,i64,f64,f64,f64,f64,f64,f64,u32,u32,u32,f64,u8,u8,u8,u8
100001,1,-1740,-1740,24835.5,23787.0,0.957782,3951.0,0.104326,8.0,0,1,0,0.0,1,0,0,0
100002,1,-606,-606,179055.0,179055.0,1.0,9251.775,0.0,24.0,0,1,0,0.0,1,0,0,0
100003,3,-746,-2341,1306309.5,1.452573e6,1.111967,56553.99,0.05003,10.0,0,3,0,0.0,1,0,0,0
100004,1,-815,-815,24282.0,20106.0,0.828021,5357.25,0.212008,4.0,0,1,0,0.0,1,0,0,0
100005,2,-315,-757,44617.5,40153.5,0.89995,4813.2,0.108964,12.0,0,1,1,0.0,0,1,0,0


In [18]:
print(bureau_final_features["SK_ID_CURR"].is_unique().all())
print(ins_features["SK_ID_CURR"].is_unique().all())
print(cc_features["SK_ID_CURR"].is_unique().all())
print(pos_features["SK_ID_CURR"].is_unique().all())
print(prev_features["SK_ID_CURR"].is_unique().all())

True
True
True
True
True


In [19]:
application_train = pl.read_csv("../data/raw/application_train.csv")
application_test = pl.read_csv("../data/raw/application_test.csv")

def assemble_master_dataset(base_application_df, bureau_df, ins_df, cc_df, pos_df, prev_df):
    """
    Sequentially joins all engineered feature tables onto a base application dataframe (train or test).
    """
    return (
        base_application_df
        .join(bureau_df, on="SK_ID_CURR", how="left")
        .join(ins_df, on="SK_ID_CURR", how="left")
        .join(cc_df, on="SK_ID_CURR", how="left")
        .join(pos_df, on="SK_ID_CURR", how="left")
        .join(prev_df, on="SK_ID_CURR", how="left")
    )

train_complete = assemble_master_dataset(
    application_train, bureau_final_features, ins_features, cc_features, pos_features, prev_features
)

test_complete = assemble_master_dataset(
    application_test, bureau_final_features, ins_features, cc_features, pos_features, prev_features
)

print(f"Train Shape: {train_complete.shape}")
print(f"Test Shape: {test_complete.shape}")

Train Shape: (307511, 184)
Test Shape: (48744, 183)
